# 03 — Exploratory data analysis

**Phase 3.** Inputs are the Parquet tables written by `src/clean.py`; nothing is read from
`data/raw/` here and nothing is written back to it.

Figures are saved to `reports/figures/`. Each one is followed by a one-line takeaway.
Covered: trends, the pandemic structural break, regional differences, concentration,
tourism intensity and pressure, correlations, seasonality, origin-destination flows, and
outliers.

Run with: `python -m jupyter nbconvert --to notebook --execute --inplace notebooks/03_eda.ipynb`

In [1]:
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Resolve ml/ whether the notebook is run from ml/ or from ml/notebooks/.
ML = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED = ML / "data" / "processed"
FIGURES = ML / "reports" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.dpi": 130, "savefig.dpi": 130, "font.size": 9,
                     "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.autolayout": True})


def save(fig, name):
    path = FIGURES / name
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    print(f"saved {path.relative_to(ML).as_posix()}")


panel = pd.read_parquet(PROCESSED / "state_year_panel.parquet")
panel["state"] = panel["state"].astype(str)
flows = pd.read_parquet(PROCESSED / "od_flows.parquet")
quarterly = pd.read_parquet(PROCESSED / "national_quarterly.parquet")

print(f"panel     {panel.shape}  {panel.year.min()}-{panel.year.max()}  {panel.state.nunique()} states")
print(f"flows     {flows.shape}  years {sorted(flows.year.unique())}")
print(f"quarterly {quarterly.shape}")

panel     (160, 21)  2016-2025  16 states
flows     (768, 4)  years [np.int64(2023), np.int64(2024), np.int64(2025)]
quarterly (21, 5)


## 1. The pandemic is the dominant feature of the series

In [2]:
national = panel.groupby("year")["visitors_000"].sum() / 1000
print(national.round(1).to_string())
print("\nyear-on-year %:")
print((national.pct_change() * 100).round(1).to_string())

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.axvspan(2019.5, 2022.5, color="0.85", zorder=0)
ax.plot(national.index, national.values, marker="o", color="#1f4e79")
ax.annotate("movement controls\n(2020-2021)", (2020.5, national.max() * 0.45),
            ha="center", fontsize=8, color="0.3")
ax.annotate("rebound\n2022", (2022, national.loc[2022] + 18), ha="center", fontsize=8, color="0.3")
ax.set_xlabel("Year"); ax.set_ylabel("Domestic visitors (millions)")
ax.set_title("Malaysia domestic visitors, 2016-2025")
save(fig, "01_national_trend.png")

year
2016    189.3
2017    205.4
2018    221.3
2019    239.1
2020    131.7
2021     66.0
2022    171.6
2023    213.7
2024    260.1
2025    290.1

year-on-year %:
year
2016      NaN
2017      8.5
2018      7.7
2019      8.1
2020    -44.9
2021    -49.9
2022    160.1
2023     24.6
2024     21.7
2025     11.5
saved reports/figures/01_national_trend.png


**Takeaway.** Domestic visitors fell from 239.1m (2019) to 66.0m (2021), then recovered to
290.1m by 2025 — so 2020-2022 is a structural break, not noise, and any model trained across
it must either flag those years or exclude them.

## 2. Every state has recovered past its 2019 level

In [3]:
wide = panel.pivot(index="year", columns="state", values="visitors_000")
indexed = wide / wide.loc[2019] * 100
recovery = indexed.loc[2025].sort_values()
print("2025 as % of 2019:")
print(recovery.round(0).to_string())

fig, ax = plt.subplots(figsize=(7.5, 4))
for state in indexed.columns:
    ax.plot(indexed.index, indexed[state], color="0.75", linewidth=0.9)
for state, colour in [(recovery.index[-1], "#c0392b"), (recovery.index[0], "#1f4e79")]:
    ax.plot(indexed.index, indexed[state], linewidth=2.2, color=colour, label=state)
ax.axhline(100, color="black", linewidth=0.8, linestyle="--")
ax.set_xlabel("Year"); ax.set_ylabel("Visitors, 2019 = 100")
ax.set_title("State recovery paths, indexed to 2019")
ax.legend(frameon=False, fontsize=8)
save(fig, "02_state_recovery.png")

2025 as % of 2019:
state
Sabah                101.0
Kedah                105.0
Selangor             108.0
Terengganu           109.0
Kelantan             110.0
Perak                112.0
Sarawak              115.0
Pulau Pinang         115.0
W.P. Labuan          115.0
Pahang               125.0
Johor                127.0
Negeri Sembilan      146.0
Melaka               149.0
W.P. Kuala Lumpur    155.0
W.P. Putrajaya       161.0
Perlis               180.0


saved reports/figures/02_state_recovery.png


**Takeaway.** All 16 states are above their 2019 level by 2025 (median 115%), so the
recovery is complete and 2023-2025 can be treated as a post-pandemic regime.

## 3. Concentration has not shifted

In [4]:
shares = {}
for year in (2019, 2025):
    s = panel[panel.year == year].set_index("state")["visitors_000"]
    shares[year] = s / s.sum() * 100
share = pd.DataFrame(shares).sort_values(2025)
for year in (2019, 2025):
    top3 = share[year].nlargest(3)
    print(f"{year}: top 3 = {top3.round(1).to_dict()}, together {top3.sum():.1f}%; "
          f"bottom 3 together {share[year].nsmallest(3).sum():.1f}%")

fig, ax = plt.subplots(figsize=(7, 4.6))
y = np.arange(len(share))
ax.hlines(y, share[2019], share[2025], color="0.75", linewidth=1.6, zorder=1)
ax.scatter(share[2019], y, s=26, color="0.55", label="2019", zorder=2)
ax.scatter(share[2025], y, s=26, color="#c0392b", label="2025", zorder=2)
ax.set_yticks(y); ax.set_yticklabels(share.index)
ax.set_xlabel("Share of national domestic visitors (%)")
ax.set_title("Where domestic visitors go: 2019 vs 2025")
ax.legend(frameon=False, fontsize=8)
save(fig, "03_concentration.png")

2019: top 3 = {'Selangor': 14.0, 'W.P. Kuala Lumpur': 9.5, 'Sabah': 9.2}, together 32.7%; bottom 3 together 1.9%
2025: top 3 = {'Selangor': 12.5, 'W.P. Kuala Lumpur': 12.1, 'Perak': 8.2}, together 32.8%; bottom 3 together 2.6%


saved reports/figures/03_concentration.png


**Takeaway.** The top three states held 32.7% of national visitors in 2019 and 32.8% in
2025, so six years including a pandemic did not redistribute domestic tourism.

## 4. Absolute size and tourism intensity are different rankings

In [5]:
latest = panel[panel.year == 2025].copy()
latest["per_resident"] = latest.visitors_000 / latest.population_000
latest = latest.sort_values("per_resident")
print(latest[["state", "visitors_000", "population_000", "per_resident"]].round(2).to_string(index=False))
print(f"\nspread: {latest.per_resident.max():.1f} (highest) vs {latest.per_resident.min():.1f} "
      f"= {latest.per_resident.max() / latest.per_resident.min():.1f}x")

fig, axes = plt.subplots(1, 2, figsize=(9, 4.4))
order_abs = latest.sort_values("visitors_000")
axes[0].barh(order_abs.state, order_abs.visitors_000 / 1000, color="#1f4e79")
axes[0].set_xlabel("Visitors (millions)"); axes[0].set_title("Absolute volume, 2025")
axes[1].barh(latest.state, latest.per_resident, color="#c0392b")
axes[1].set_xlabel("Visitors per resident"); axes[1].set_title("Tourism intensity, 2025")
save(fig, "04_volume_vs_intensity.png")

            state  visitors_000  population_000  per_resident
            Johor      18196.97          4204.1          4.33
         Selangor      36376.44          7408.7          4.91
            Sabah      22361.17          3754.5          5.96
      W.P. Labuan        604.37           100.9          5.99
         Kelantan      12062.02          1906.9          6.33
            Kedah      15607.88          2227.1          7.01
          Sarawak      22721.53          2528.9          8.98
            Perak      23642.06          2573.8          9.19
     Pulau Pinang      17717.98          1804.3          9.82
       Terengganu      15462.28          1245.8         12.41
           Perlis       3755.68           297.0         12.65
           Pahang      23161.17          1676.8         13.81
  Negeri Sembilan      19356.54          1243.9         15.56
W.P. Kuala Lumpur      35059.93          2075.2         16.89
           Melaka      20832.20          1052.0         19.80
   W.P. 

saved reports/figures/04_volume_vs_intensity.png


**Takeaway.** Ranking by intensity reorders the map entirely — W.P. Putrajaya (26.1 visitors
per resident) and Melaka (19.8) lead, while Selangor and Johor, first and fourth by volume,
sit last at 4.9 and 4.3 — a 6.0x spread that a volume-only view hides.

## 5. Accommodation pressure

In [6]:
latest["per_room"] = latest.visitors_000 * 1000 / latest.rooms
pressure = latest.dropna(subset=["per_room"]).sort_values("per_room")
print(pressure[["state", "visitors_000", "rooms", "per_room"]].round(0).to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4.4))
ax.barh(pressure.state, pressure.per_room, color="#7d3c98")
ax.set_xlabel("Visitors per hotel room, 2025")
ax.set_title("Visitor load per unit of accommodation capacity")
save(fig, "05_accommodation_pressure.png")

            state  visitors_000   rooms  per_room
      W.P. Labuan         604.0  1697.0     356.0
            Johor       18197.0 31184.0     584.0
     Pulau Pinang       17718.0 24034.0     737.0
W.P. Kuala Lumpur       35060.0 47177.0     743.0
           Pahang       23161.0 25624.0     904.0
            Sabah       22361.0 23420.0     955.0
          Sarawak       22722.0 21142.0    1075.0
           Melaka       20832.0 18223.0    1143.0
            Kedah       15608.0 13154.0    1187.0
       Terengganu       15462.0 11139.0    1388.0
            Perak       23642.0 16450.0    1437.0
         Selangor       36376.0 24965.0    1457.0
   W.P. Putrajaya        3146.0  1769.0    1779.0
  Negeri Sembilan       19357.0  9536.0    2030.0
         Kelantan       12062.0  4360.0    2767.0
           Perlis        3756.0  1245.0    3017.0
saved reports/figures/05_accommodation_pressure.png


**Takeaway.** Visitors per hotel room ranges from 3,017 in Perlis and 2,767 in Kelantan down
to 584 in Johor and 356 in W.P. Labuan — but most domestic visitors are excursionists rather
than overnight tourists, so this measures visitor load against capacity, not occupancy.

## 6. Structural predictors are strongly log-linear in visitors

In [7]:
structural = panel[panel.in_structural_sample & (panel.year >= 2017)].copy()
print(f"structural sample: {len(structural)} rows, years {sorted(structural.year.unique())}")

candidates = ["population_000", "gdp_total_rm_mn", "gdp_services_rm_mn", "rooms", "hotels",
              "lf_employed_000", "p_rate", "u_rate", "cpi_accom_food", "cpi_recreation"]
linear = structural[["visitors_000"] + candidates].corr()["visitors_000"].drop("visitors_000")
print("\nPearson correlation with visitors (levels):")
print(linear.sort_values(ascending=False).round(3).to_string())

logged = structural.dropna(subset=["rooms", "gdp_total_rm_mn", "lf_employed_000"]).copy()
for column in ["visitors_000", "population_000", "gdp_total_rm_mn", "gdp_services_rm_mn",
               "rooms", "lf_employed_000"]:
    logged["log_" + column] = np.log(logged[column])
log_cols = [c for c in logged.columns if c.startswith("log_") and c != "log_visitors_000"]
log_corr = logged[["log_visitors_000"] + log_cols].corr()["log_visitors_000"].drop("log_visitors_000")
print("\nPearson correlation in logs:")
print(log_corr.sort_values(ascending=False).round(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, xcol, label in [(axes[0], "log_population_000", "log population ('000)"),
                        (axes[1], "log_rooms", "log hotel rooms")]:
    ax.scatter(logged[xcol], logged["log_visitors_000"], s=18, alpha=0.75, color="#1f4e79")
    slope, intercept = np.polyfit(logged[xcol], logged["log_visitors_000"], 1)
    xs = np.linspace(logged[xcol].min(), logged[xcol].max(), 50)
    ax.plot(xs, slope * xs + intercept, color="#c0392b", linewidth=1.4)
    r = logged[[xcol, "log_visitors_000"]].corr().iloc[0, 1]
    ax.set_xlabel(label); ax.set_ylabel("log visitors ('000)")
    ax.set_title(f"r = {r:.3f},  slope = {slope:.2f}")
fig.suptitle("Structural sample (2017-2019, 2023-2025), log-log", y=1.02)
save(fig, "06_loglog_structure.png")

structural sample: 96 rows, years [np.int16(2017), np.int16(2018), np.int16(2019), np.int16(2023), np.int16(2024), np.int16(2025)]

Pearson correlation with visitors (levels):
gdp_total_rm_mn       0.794
population_000        0.770
lf_employed_000       0.766
rooms                 0.753
gdp_services_rm_mn    0.742
hotels                0.659
p_rate                0.391
cpi_accom_food        0.209
cpi_recreation        0.090
u_rate               -0.121

Pearson correlation in logs:
log_population_000        0.907
log_lf_employed_000       0.906
log_rooms                 0.859
log_gdp_total_rm_mn       0.844
log_gdp_services_rm_mn    0.781


saved reports/figures/06_loglog_structure.png


**Takeaway.** In logs the structural relationships are strong and near-linear —
population 0.907, employment 0.906, hotel rooms 0.859, GDP 0.844 — which is what makes a
log-linear expected-demand model defensible rather than arbitrary. (The sample printed
above is 96 state-years; 93 of those also have every core predictor present, which is the
number `clean.py` reports and the number the model will actually train on.)

## 7. Correlation structure among the predictors

In [8]:
matrix = structural[["visitors_000"] + candidates].corr()
fig, ax = plt.subplots(figsize=(6.4, 5.6))
im = ax.imshow(matrix, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(matrix))); ax.set_xticklabels(matrix.columns, rotation=90, fontsize=7)
ax.set_yticks(range(len(matrix))); ax.set_yticklabels(matrix.columns, fontsize=7)
for i in range(len(matrix)):
    for j in range(len(matrix)):
        ax.text(j, i, f"{matrix.iloc[i, j]:.2f}", ha="center", va="center", fontsize=6,
                color="white" if abs(matrix.iloc[i, j]) > 0.6 else "black")
ax.grid(False)
fig.colorbar(im, ax=ax, shrink=0.8)
ax.set_title("Correlations, structural sample")
save(fig, "07_correlation_matrix.png")

size_block = ["population_000", "gdp_total_rm_mn", "gdp_services_rm_mn", "lf_employed_000", "rooms"]
off_diagonal = matrix.loc[size_block, size_block].values[np.triu_indices(len(size_block), 1)]
print(f"pairwise correlation among the size variables: min {off_diagonal.min():.3f}, "
      f"max {off_diagonal.max():.3f}")

saved reports/figures/07_correlation_matrix.png
pairwise correlation among the size variables: min 0.541, max 0.992


**Takeaway.** Population, GDP, services GDP, employment and rooms all measure state size
and are collinear with each other (pairwise correlation 0.541 to 0.992), so a regularised
or tree-based model is needed and individual coefficients must not be read as independent
effects.

## 8. Seasonality is only visible at national quarterly level

In [9]:
q = quarterly.dropna(subset=["visitors_000"]).copy()
q["label"] = q.year.astype(str) + " Q" + q.quarter.astype(str)
post = q[q.year >= 2023]
seasonal = post.groupby("quarter")["visitors_000"].mean()
seasonal_index = seasonal / seasonal.mean() * 100
print("mean quarterly visitors 2023 onwards ('000):")
print(seasonal.round(0).to_string())
print("\nseasonal index (mean quarter = 100):")
print(seasonal_index.round(1).to_string())

fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.8))
axes[0].plot(range(len(q)), q.visitors_000 / 1000, marker="o", markersize=3, color="#1f4e79")
step = 2
axes[0].set_xticks(range(0, len(q), step))
axes[0].set_xticklabels(q.label.iloc[::step], rotation=90, fontsize=7)
axes[0].set_ylabel("Visitors (millions)"); axes[0].set_title("National quarterly visitors")
axes[1].bar(seasonal_index.index, seasonal_index.values, color="#1f4e79")
axes[1].axhline(100, color="black", linewidth=0.8, linestyle="--")
axes[1].set_xlabel("Quarter"); axes[1].set_ylabel("Index (mean quarter = 100)")
axes[1].set_title("Seasonal pattern, 2023 onwards")
save(fig, "08_quarterly_seasonality.png")

mean quarterly visitors 2023 onwards ('000):
quarter
1    63055.0
2    65825.0
3    64342.0
4    65293.0

seasonal index (mean quarter = 100):
quarter
1     97.6
2    101.9
3     99.6
4    101.0
saved reports/figures/08_quarterly_seasonality.png


**Takeaway.** Quarterly variation is mild — the seasonal index runs only from 97.6 (Q1) to
101.9 (Q2) — and the state-level data is annual anyway, so seasonality cannot be modelled
per state. This is a limitation to state plainly, not a feature to engineer.

## 9. Domestic tourism is becoming less local

In [10]:
rows = []
for year, group in flows.groupby("year"):
    within = group[group.origin == group.destination].tourists_000.sum()
    rows.append({"year": year, "within_state_pct": within / group.tourists_000.sum() * 100,
                 "total_000": group.tourists_000.sum()})
locality = pd.DataFrame(rows)
print(locality.round(1).to_string(index=False))

top = (flows[(flows.year == 2025) & (flows.origin != flows.destination)]
       .nlargest(10, "tourists_000"))
print("\ntop 10 inter-state flows, 2025 ('000 tourists):")
print(top.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
axes[0].plot(locality.year, locality.within_state_pct, marker="o", color="#1f4e79")
axes[0].set_xticks(locality.year); axes[0].set_ylim(0, 45)
axes[0].set_xlabel("Year"); axes[0].set_ylabel("% of tourists staying in their own state")
axes[0].set_title("Share of within-state tourism")
pairs = (top.origin + " -> " + top.destination)[::-1]
axes[1].barh(pairs, top.tourists_000[::-1], color="#c0392b")
axes[1].set_xlabel("Tourists ('000), 2025"); axes[1].set_title("Largest inter-state flows")
axes[1].tick_params(axis="y", labelsize=7)
save(fig, "09_od_flows.png")

 year  within_state_pct  total_000
 2023              36.4    79559.4
 2024              25.6    93445.3
 2025              22.6   106525.3

top 10 inter-state flows, 2025 ('000 tourists):
 year            origin       destination  tourists_000
 2025          Selangor             Perak    3116.56300
 2025          Selangor            Melaka    2725.94100
 2025          Selangor            Pahang    2517.56353
 2025          Selangor      Pulau Pinang    2362.70000
 2025 W.P. Kuala Lumpur             Perak    2017.80200
 2025          Selangor             Johor    1995.97700
 2025          Selangor   Negeri Sembilan    1980.10700
 2025          Selangor W.P. Kuala Lumpur    1819.35000
 2025             Perak W.P. Kuala Lumpur    1771.24400
 2025             Perak          Selangor    1716.46700


saved reports/figures/09_od_flows.png


**Takeaway.** The share of tourists staying within their own state fell from 36.4% in 2023 to
22.6% in 2025, and every one of the ten largest inter-state flows originates in Selangor or
W.P. Kuala Lumpur — the Klang Valley is the country's single domestic source market.

## 10. Outliers and volatility

In [11]:
growth = panel.sort_values(["state", "year"]).copy()
growth["yoy"] = growth.groupby("state")["visitors_000"].pct_change() * 100
normal = growth[(growth.year >= 2017) & (~growth.year.isin([2020, 2021, 2022]))]
print("largest increases outside the pandemic years:")
print(normal.nlargest(5, "yoy")[["state", "year", "yoy"]].round(1).to_string(index=False))
print("\nlargest decreases:")
print(normal.nsmallest(5, "yoy")[["state", "year", "yoy"]].round(1).to_string(index=False))

volatility = (normal.groupby("state")["yoy"].std()
              .to_frame("yoy_sd")
              .join(panel[panel.year == 2025].set_index("state")["visitors_000"])
              .dropna().sort_values("visitors_000"))
print("\nyear-on-year volatility by state size:")
print(volatility.round(1).to_string())

fig, ax = plt.subplots(figsize=(6.6, 4.2))
ax.scatter(volatility.visitors_000 / 1000, volatility.yoy_sd, s=28, color="#1f4e79")
for state, row in volatility.iterrows():
    if row.yoy_sd > 20 or row.visitors_000 > 30000:
        ax.annotate(state, (row.visitors_000 / 1000, row.yoy_sd), fontsize=7,
                    xytext=(3, 3), textcoords="offset points")
ax.set_xlabel("Visitors in 2025 (millions)")
ax.set_ylabel("Std dev of year-on-year % change")
ax.set_title("Smaller states are noisier (pandemic years excluded)")
save(fig, "10_volatility.png")

largest increases outside the pandemic years:
         state  year  yoy
        Perlis  2024 65.3
        Perlis  2018 52.5
W.P. Putrajaya  2023 45.5
   W.P. Labuan  2018 43.0
      Kelantan  2024 39.3

largest decreases:
            state  year   yoy
            Perak  2018 -12.7
      W.P. Labuan  2019  -3.8
           Perlis  2019  -3.1
           Perlis  2017   0.3
W.P. Kuala Lumpur  2018   0.6

year-on-year volatility by state size:
                   yoy_sd  visitors_000
state                                  
W.P. Labuan          18.9         604.4
W.P. Putrajaya       18.9        3146.5
Perlis               28.0        3755.7
Kelantan             12.5       12062.0
Terengganu            7.4       15462.3
Kedah                 6.8       15607.9
Pulau Pinang         12.2       17718.0
Johor                 9.0       18197.0
Negeri Sembilan       9.8       19356.5
Melaka               12.0       20832.2
Sabah                 9.7       22361.2
Sarawak               5.1       22721.

saved reports/figures/10_volatility.png


**Takeaway.** The largest non-pandemic swings belong to the smallest states — Perlis
+65.3% in 2024 and +52.5% in 2018 — and the three smallest states have the three highest
year-on-year standard deviations (Perlis 28.0, W.P. Labuan and W.P. Putrajaya 18.9 each)
against 5.1 for Sarawak. Percentage-error metrics will therefore be dominated by small
states, so evaluation must report absolute error as well.

## Figures produced

| File | Shows |
|---|---|
| `01_national_trend.png` | national visitors 2016-2025 with the pandemic marked |
| `02_state_recovery.png` | state paths indexed to 2019 = 100 |
| `03_concentration.png` | share of national visitors, 2019 vs 2025 |
| `04_volume_vs_intensity.png` | absolute volume against visitors per resident |
| `05_accommodation_pressure.png` | visitors per hotel room |
| `06_loglog_structure.png` | log visitors against log population and log rooms |
| `07_correlation_matrix.png` | correlations among candidate predictors |
| `08_quarterly_seasonality.png` | national quarterly series and seasonal index |
| `09_od_flows.png` | within-state share over time, largest inter-state flows |
| `10_volatility.png` | year-on-year volatility against state size |